In [ ]:
# ============================================================================
# CELL 1 — Anonymise raw files into memory
# ============================================================================
# Reads all raw Polar JSONs from INPUT_DIR, strips identifiers and unwanted
# fields, and stores the anonymised dicts in `anon_sessions` (a list).
# Nothing is written to disk yet.
# Re-run only when the source files change.

import hashlib
import json
import logging
import math
from datetime import datetime

INPUT_DIR  = "/Volumes/multisport_training/default/training_data"
OUTPUT_DIR = "/Volumes/multisport_training/default/training_data_anon"

HASH_SALT = "multisport-training-anon-v1"

KEEP_STREAMS = {
    "HEART_RATE", "SPEED", "CADENCE", "ALTITUDE", "DISTANCE", "TEMPERATURE",
}

KEEP_THRESHOLDS = {
    "maximumHeartRate":         "hr_max",
    "restingHeartRate":         "hr_resting",
    "aerobicThreshold":         "aerobic_threshold",
    "anaerobicThreshold":       "anaerobic_threshold",
    "vo2Max":                   "vo2max",
    "functionalThresholdPower": "ftp",
    "maximumAerobicSpeedKmh":   "mas_kmh",
    "maximumAerobicPower":      "map_watts",
}

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("anonymize")


def hash_id(raw, prefix="id"):
    """Deterministic short hash. Different prefixes give different namespaces
    so a session_id and user_id derived from the same input string differ."""
    h = hashlib.sha256((HASH_SALT + ":" + prefix + ":" + str(raw)).encode()).hexdigest()
    return h[:16]


def derive_user_id(data):
    """
    Derive a stable anonymous user_id from the raw session.
    Uses deviceId if present (one watch = one user). Falls back to "unknown"
    so files without a deviceId don't crash the run.
    """
    device_id = data.get("deviceId")
    if device_id:
        return hash_id(device_id, prefix="user")
    return "unknown_user"


def clean_value(v):
    """NaN string, NaN float, or None -> None. Real numbers pass through."""
    if v is None:
        return None
    if isinstance(v, str):
        return None
    if isinstance(v, float) and math.isnan(v):
        return None
    return v


def clean_stream(values):
    return [clean_value(v) for v in values]


def stream_has_data(values):
    return any(v is not None for v in values)


def parse_pause_intervals(pause_times, start_iso):
    if not pause_times:
        return []
    try:
        start = datetime.fromisoformat(start_iso)
    except (ValueError, TypeError):
        return []
    out = []
    for p in pause_times:
        try:
            s = datetime.fromisoformat(p["startTime"])
            e = datetime.fromisoformat(p["endTime"])
            out.append([
                int((s - start).total_seconds() * 1000),
                int((e - start).total_seconds() * 1000),
            ])
        except (KeyError, ValueError, TypeError):
            pass
    return out


def extract_thresholds(physical_info):
    if not physical_info:
        return {}
    return {
        new_key: physical_info[orig_key]
        for orig_key, new_key in KEEP_THRESHOLDS.items()
        if orig_key in physical_info and physical_info[orig_key] is not None
    }


def anonymize_exercise(ex, session_anon_id, ex_index):
    samples_raw = ex.get("samples", {}).get("samples", [])
    streams = {}
    interval_ms = None
    for s in samples_raw:
        stype = s.get("type", "")
        if stype not in KEEP_STREAMS:
            continue
        cleaned = clean_stream(s.get("values", []))
        if not stream_has_data(cleaned):
            continue
        streams[stype.lower()] = cleaned
        if interval_ms is None:
            interval_ms = s.get("intervalMillis")

    if not streams:
        return None

    sport_id = ex.get("sport", {}).get("id")
    return {
        "exercise_index": ex_index,
        "exercise_id": hash_id(ex.get("identifier", {}).get("id", f"{session_anon_id}-{ex_index}"), prefix="ex"),
        "start_time_iso": ex.get("startTime"),
        "sport_id": sport_id,
        # sport_name intentionally NOT set here -- assigned in cell 2
        "samples": {"interval_ms": interval_ms, "streams": streams},
        "pause_intervals_ms": parse_pause_intervals(
            ex.get("pauseTimes", []) or [], ex.get("startTime", "")
        ),
    }


def anonymize_session(data):
    raw_id = data.get("identifier", {}).get("id")
    if raw_id is None:
        return None

    session_anon_id = hash_id(raw_id, prefix="session")
    user_id = derive_user_id(data)

    exercises = []
    for i, ex in enumerate(data.get("exercises", []) or []):
        anon_ex = anonymize_exercise(ex, session_anon_id, i)
        if anon_ex is not None:
            exercises.append(anon_ex)

    if not exercises:
        return None

    sport_id = data.get("sport", {}).get("id")
    return {
        "user_id": user_id,
        "session_id": session_anon_id,
        "start_time_iso": data.get("startTime"),
        "timezone_offset_minutes": data.get("timezoneOffsetMinutes"),
        "sport_id": sport_id,
        # sport_name intentionally NOT set here -- assigned in cell 2
        "athlete_thresholds": extract_thresholds(data.get("physicalInformation", {}) or {}),
        "exercises": exercises,
    }


def list_json_files(root):
    out = []
    stack = [root]
    while stack:
        current = stack.pop()
        try:
            entries = dbutils.fs.ls(current)
        except Exception as e:
            log.warning("Cannot list %s: %s", current, e)
            continue
        for e in entries:
            if e.isDir():
                stack.append(e.path)
            elif e.name.endswith(".json"):
                out.append(e.path)
    return out


def read_text(path):
    return dbutils.fs.head(path, 64 * 1024 * 1024)


# Run the anonymisation
files = list_json_files(INPUT_DIR)
log.info("Found %d JSON files under %s", len(files), INPUT_DIR)

anon_sessions = []
skipped = 0
for fp in files:
    try:
        text = read_text(fp)
        raw = json.loads(text, parse_constant=lambda x: None)
    except Exception as e:
        log.error("Cannot read/parse %s: %s", fp, e)
        skipped += 1
        continue
    anon = anonymize_session(raw)
    if anon is None:
        skipped += 1
        continue
    anon_sessions.append(anon)

# Report on user IDs found
user_counts = {}
for s in anon_sessions:
    uid = s["user_id"]
    user_counts[uid] = user_counts.get(uid, 0) + 1

log.info("Anonymised %d sessions in memory (skipped %d)", len(anon_sessions), skipped)
log.info("Distinct user IDs: %d", len(user_counts))
for uid, count in sorted(user_counts.items(), key=lambda x: -x[1]):
    log.info("  user_id=%s: %d session(s)", uid, count)

In [ ]:
# ============================================================================
# CELL 2 — Assign sport names and surface unknown sport IDs
# ============================================================================
# Reads the in-memory `anon_sessions` from cell 1, applies SPORT_NAMES, and
# reports any IDs that aren't recognised. For each unknown ID it shows the
# session_id and date of the most recent session using it, so you can find
# it in Polar Flow and identify the sport.
#
# Edit SPORT_NAMES below as you identify new IDs, then re-run this cell.
# Cell 1 doesn't need to re-run.

# Polar JSON sport_id to readable name.
# Verified IDs only -- add new entries here as you identify them in Polar Flow.
SPORT_NAMES: dict[str, str] = {
    "1":   "RUNNING",         # confirmed: outdoor run, cadence ~77 spm, altitude variation
    "23":  "POOL_SWIMMING",   # confirmed: 400m exact, all-zero cadence, constant 25C water temp
    "2":  "CYCLING", 
    "18": "INDOOR_CYCLING",
   "15": "STRENGTH_TRAINING",
   "105": "OPEN_WATER_SWIMMING",
   "3": "WALKING",
   "82": "MULTISPORT",
   "7": "ALPINE_SKI",
   "127": "MOBILITY",
   "11": "HIKING",
   "68": "TRIATHLON",
   "83": "OTHER_INDOOR",
   "5": "MOUNTAIN_BIKING",
   "27": "TRAIL_RUNNING",
   "34": "HIIT",
   "17": "INDOOR_RUNNING",
   "16":"OTHER_OUTDOOR"
}


def sport_name_for(sport_id):
    if sport_id is None:
        return "UNKNOWN_NONE"
    sid = str(sport_id)
    return SPORT_NAMES.get(sid, f"UNKNOWN_{sid}")


# Apply names in place to anon_sessions
for s in anon_sessions:
    s["sport_name"] = sport_name_for(s.get("sport_id"))
    for ex in s["exercises"]:
        ex["sport_name"] = sport_name_for(ex.get("sport_id"))

# Build a report of unknown IDs with examples (newest session per ID)
unknown = {}  # sport_id -> {"count": int, "newest": session_dict}
for s in anon_sessions:
    sid = s.get("sport_id")
    if sid is None or str(sid) in SPORT_NAMES:
        continue
    key = str(sid)
    entry = unknown.setdefault(key, {"count": 0, "newest": None})
    entry["count"] += 1
    # Track the newest session by start_time_iso (string-sortable in ISO format)
    if entry["newest"] is None or (s.get("start_time_iso") or "") > (entry["newest"].get("start_time_iso") or ""):
        entry["newest"] = s

# Print the report
print(f"Total sessions: {len(anon_sessions)}")
known_count = sum(1 for s in anon_sessions if str(s.get("sport_id")) in SPORT_NAMES)
print(f"With known sport_name: {known_count}")
print(f"With unknown sport_id: {len(anon_sessions) - known_count}")
print()

if unknown:
    print(f"Unknown sport IDs ({len(unknown)} distinct):")
    print(f"{'sport_id':<10} {'count':<8} {'newest_session_id':<20} {'newest_start_time'}")
    print("-" * 70)
    for sid, info in sorted(unknown.items(), key=lambda x: -x[1]["count"]):
        n = info["newest"]
        print(f"{sid:<10} {info['count']:<8} {n['session_id']:<20} {n.get('start_time_iso')}")
    print()
    print("To identify each ID:")
    print("  1. Open https://flow.polar.com and find the session at the date/time shown")
    print("  2. The sport name is in the session header")
    print("  3. Add to SPORT_NAMES above and re-run this cell")
else:
    print("All sport IDs recognised.")


In [ ]:
# ============================================================================
# CELL 3 — Clean OUTPUT_DIR and write anonymised files
# ============================================================================
# Deletes all .json files in OUTPUT_DIR before writing, so stale files from
# previous runs don't accumulate. (E.g. if you change HASH_SALT or remove a
# source file, the old anonymised version would otherwise linger.)

# Safety check: refuse to wipe the output directory if there's nothing to
# replace it with. Catches the case where cell 1 failed silently or wasn't
# run yet.
if not anon_sessions:
    raise RuntimeError(
        "anon_sessions is empty -- refusing to wipe OUTPUT_DIR. "
        "Re-run cell 1 first."
    )

# Make sure the directory exists
dbutils.fs.mkdirs(OUTPUT_DIR)

# Delete existing .json files (only json, leave other files alone)
deleted = 0
try:
    existing = dbutils.fs.ls(OUTPUT_DIR)
except Exception as e:
    log.warning("Cannot list %s for cleanup: %s", OUTPUT_DIR, e)
    existing = []

for entry in existing:
    if entry.name.endswith(".json"):
        try:
            dbutils.fs.rm(entry.path)
            deleted += 1
        except Exception as e:
            log.error("Cannot delete %s: %s", entry.path, e)

log.info("Deleted %d existing .json file(s) from %s", deleted, OUTPUT_DIR)

# Write fresh files
written = 0
failed = 0
for s in anon_sessions:
    out_path = f"{OUTPUT_DIR.rstrip('/')}/session_{s['session_id']}.json"
    try:
        dbutils.fs.put(
            out_path,
            json.dumps(s, separators=(",", ":"), allow_nan=False),
            overwrite=True,
        )
        written += 1
    except Exception as e:
        log.error("Cannot write %s: %s", out_path, e)
        failed += 1

log.info("Wrote %d files to %s (failed %d)", written, OUTPUT_DIR, failed)